# 10. Chatbot RAG Agent 구현

**OpenAI Agents SDK**로 채팅 에이전트를 단계별로 확장합니다.
도구 없는 단순 챗봇에서 시작해 날씨/웹검색/RAG 도구를 하나씩 붙여가며 완성합니다.

```
1. 기본 대화         Agent + Runner.run()
      ↓
2. 스트리밍          Runner.run_streamed() + 토큰 단위 출력
      ↓
3. 도구 추가         + get_weather + WebSearchTool
      ↓
4. RAG 추가          + FileSearchTool (Vector Store)
      ↓
5. 웹 UI            → Streamlit app.py
```

| 단계 | 내용 | 핵심 API / 도구 |
|------|------|-----------------|
| 1. 비동기 방식 호출 | 도구 없이 Agent + Runner로 대화 구현 (메시지 유지) | `await Runner.run()` |
| 2. Streaming 방식 호출 | 동일 에이전트를 토큰 단위 실시간 출력으로 실행 | `Runner.run_streamed()`, `ResponseTextDeltaEvent` |
| 3. 도구를 추가한 Chatbot | 날씨 조회 + 웹 검색 도구를 에이전트에 연결 | `@function_tool` get_weather, `WebSearchTool()` |
| 4. RAG 도구 추가 | OpenAI Vector Store에 FAQ/KB 업로드 후 검색 도구 연결 | `FileSearchTool(vector_store_ids=[...])` |
| 5. 웹 UI 연결 | 동일 에이전트를 Streamlit 웹 앱으로 구현한 예제 안내 | `10_chat-app-rag/app.py` |

In [2]:
import openai

from dotenv import load_dotenv
load_dotenv() 

Model = "gpt-5-nano"

### 비동기 방식 호출
도구 없이 Agent + Runner로 대화 구현 

In [3]:
from agents import Agent, Runner  

messages = []  # 지금까지의 대화 내용을 저장할 리스트

agent = Agent(
    model=Model, 
    name="여행 에이전트",
    instructions="당신은 훌륭한 여행 에이전트입니다. 사용자와 대화하면서 여행 계획을 도와주세요."
)


while True:
    # 사용자 입력 받기
    user_input = input("\n사용자: ")
    
    # 'exit' 입력 시 종료
    if user_input == "exit":
        print("Bye")
        break
    
    # 현재 사용자의 발화를 대화 기록에 추가
    messages.append({"role": "user", "content": user_input})

    response = await Runner.run(agent, input=messages)
    
    # 모델의 응답을 대화 기록에 추가 (assistant 역할)
    messages.append({"role": "assistant", "content": response.final_output})
    
    # 모델의 최종 응답 출력
    print(f"\n여행 에이전트: {response.final_output}")


사용자:  안녕, 난 길동이야



여행 에이전트: 안녕하세요, 길동님! 여행 계획 도와드릴게요. 먼저 몇 가지 정보를 알려주시면 맞춤형 일정과 견적을 바로 만들어 드리겠습니다.

간단 Q&A
- 여행 목적/테마: 어떤 테마를 원하시나요? (문화/맛집/자연/휴양/가족여행/커플 여행 등)
- 기간 및 날짜: 원하시는 출발 날짜와 여행 기간은 어떻게 되나요?
- 예산: 1인당 또는 전체 예산은 어느 정도로 잡고 계신가요?
- 인원: 몇 명이 함께 가나요? 아이 유무도 알려주시면 좋아요.
- 선호 지역: 가고 싶은 나라나 도시가 있나요? 아니면 특정 분위기(해변, 도시, 산악 등)가 있나요?
- 숙소 스타일: 호텔/리조트/에어비앤비/리조트형 독채 중 선호가 있나요?
- 교통 선호: 항공권 위주, 기차/렌트카 병행, 도심 이동은 어떤 편을 원하시나요?
- 활동/관심사: 하이킹, 쇼핑, 음식 탐방, 관광지 위주 등 특별히 원하시는 활동이 있나요?
- 제약/요구사항: 식단 알레르기, 접근성 필요, 건강 관련 주의사항 등이 있나요?
- 비자/여권 상태: 필요하신지 여부를 알려주시면 비자 체크도 도와드릴게요.
- 꼭 가고 싶은 곳/피하고 싶은 곳: 있어으면 적어 주세요.

다음 단계 제안
- 위 정보를 주시면 제가 바로 2~3개의 맞춤 시나리오(일정 예시 + 예산 추정)와 함께 단순 견적을 드립니다.
- 원하시면 계절에 맞는 인기 목적지 3곳도 먼저 제안해 드릴게요. 예를 들면:
  - 봄/가을: 국내 제주나 강원도, 동남아의 합리적 가격대 지역
  - 여름: 유럽 해변, 동남아 휴양지, 일본 홋카이도/가나가와 지역
  - 겨울: 일본 홋카이도 스키 리조트, 동유럽 겨울 도시 탐방
- 원하시면 출발지(예: 서울/부산) 기준으로 항공권 가격대도 같이 비교해 드려요.

다 어떤 형태로 시작해볼까요? 답변 주시면 바로 진행해서 첫안 제시해 드리겠습니다.



사용자:  내 이름이 뭐였지?



여행 에이전트: 네, 길동님 맞습니다! 기억하고 있었어요.

이제 여행 계획을 본격적으로 시작해볼게요. 간단 Q&A에 답해 주시면 2~3개의 맞춤 시나리오(일정 예시 + 예산 추정)와 함께 견적도 바로 드리겠습니다. 원하시면 먼저 계절별 인기 목적지 3곳을 제안해 드릴 수도 있어요.

무엇부터 시작하고 싶으신가요?
- 1) Q&A 형태로 자세한 정보 입력
- 2) 먼저 계절별 인기 목적지 3곳 제안 받기

선호를 알려주시면 바로 진행하겠습니다.



사용자:  exit


Bye


### Streaming 방식 호출

In [4]:
#  모델의 답변이 완성될 때까지 기다리지 않고, 한 토큰씩 실시간으로 출력되는 "스트리밍" 방식
from openai.types.responses import ResponseTextDeltaEvent

messages = []  

agent = Agent(
    model=Model,
    name="여행 에이전트",
    instructions="당신은 훌륭한 여행 에이전트입니다. 사용자와 대화하면서 여행 계획을 도와주세요.",
)

while True:
    user_input = input("\n사용자: ")
    
    if user_input == "exit":
        print("Bye")
        break

    # 현재 사용자 발화를 messages 리스트에 추가
    messages.append({"role": "user", "content": user_input})

    # 에이전트의 답변 출력 시작
    print("\n여행 에이전트: ", end="", flush=True)

    # run_streamed: 모델이 응답을 실시간으로 보낼 수 있도록 함
    # 결과(result)는 이벤트 스트림(event stream)을 포함하며,
    # 이를 async for 루프를 통해 한 토큰씩 처리합니다.
    result = Runner.run_streamed(agent, input=messages)
    full_response = ""  # 전체 응답을 누적할 변수

    # ResponseTextDeltaEvent: 모델이 생성 중인 텍스트 일부를 전달하는 이벤트 타입
    # event.type == "raw_response_event" 일 때,
    # event.data.delta 에는 모델이 방금 생성한 텍스트 조각이 들어 있음
    async for event in result.stream_events():
        if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
            delta = event.data.delta or ""   # 새로 들어온 텍스트 조각
            print(delta, end="", flush=True) # 화면에 즉시 출력
            full_response += delta           # 전체 답변 문자열로 누적

    # 모델의 전체 응답을 대화 기록에 추가
    messages.append({"role": "assistant", "content": full_response})


사용자:  안녕, 난 길동이야



여행 에이전트: 반가워요, 길동님! 여행 계획 도와드릴게요.

먼저 몇 가지 정보를 알려주시면 맞춤 계획을 바로 만들어 드리겠습니다.
- 여행 기간과 톤(예: 3일 단기, 5–7일 긴 여정)
- 출발 도시
- 방문하고 싶은 나라/도시 또는 관심 분야(도시 음식, 자연, 해변, 역사 문화, 쇼핑 등)
- 예산 범위(1인당 대략 어느 정도인지)
- 동반자 수와 건강/활동 제약(걷기 많이 필요한지 등)
- 숙소 선호(호텔, 에어비앤비, 리조트) 및 등급(예산/중간/고급)
- 특별한 요청이나 제약(비자, 알레르기, 식단, 이동 수단 선호 등)

참고로, 바로 시작하고 싶으시면 2–3개의 후보지에 대해 5일짜리 샘플 일정을 간단히 제안해 드릴 수 있어요. 예시 몇 가지를 미리 드리자면:
- 제주도 4일: 자연 풍광, 맛집, 해변 여유
- 도쿄 5일: 도시 문화+맛집+주요 명소
- 동남아 6일: 해변+문화 체험(다낭/발리 근처 등)

원하시는 방향을 알려주시면 곧바로 맞춤 계획안과 예산표까지 만들어 드릴게요. 어떤지 말씀해 주세요!


사용자:  exit


Bye


### 도구를 추가한 Chatbot Agent

WebSearchTool - 모델이 응답을 생성하기 전에 웹에서 최신 정보를 검색할 수 있도록 허용합니다.  
https://developers.openai.com/api/docs/guides/tools-web-search

In [5]:
import requests
from agents import Agent, Runner, function_tool, WebSearchTool
from openai.types.responses import ResponseTextDeltaEvent

@function_tool
def get_weather(위도: float, 경도: float) -> str:
    print(위도, 경도)
    print(f"Weather 함수 실행 - 도시: {위도, 경도}")
    latitude = 위도
    longitude = 경도
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m")
    data = response.json()
    return data['current']['temperature_2m']

messages = []

agent = Agent(
    model=Model,
    name="여행 에이전트",
    instructions="당신은 훌륭한 여행 에이전트입니다. 사용자와 대화하면서 여행 계획을 도와주세요.",
    tools=[get_weather, WebSearchTool()]
)

while True:
    user_input = input("\n사용자: ")

    if user_input == "exit":
        print("Bye")
        break

    messages.append({"role": "user", "content": user_input})

    print("\n여행 에이전트: ", end="", flush=True)

    result = Runner.run_streamed(agent, input=messages)
    full_response = ""

    async for event in result.stream_events():
        if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
            delta = event.data.delta or ""
            print(delta, end="", flush=True)
            full_response += delta

    messages.append({"role": "assistant", "content": full_response})


사용자:  지금 서울의 기온을 알려줘



여행 에이전트: 지금 서울의 기온은 13°C(화씨 55°F), 현재 맑은 상태입니다. 데이터 출처: 기상청(또는 관련 최신 기상정보). 원하시면 1시간 단위 예보나 주간 예보도 알려드릴게요. 


사용자:  exit


Bye


### RAG(검색 증강 생성) 도구 추가

**RAG**는 지식 베이스(FAQ, 문서)를 검색해 사용자 질문에 관련된 내용을 맥락으로 제공하는 방식입니다.
여기서는 **OpenAI 관리형 Vector Store**에 FAQ와 지식 베이스 파일을 업로드하고,
에이전트의 **`FileSearchTool`** 로 연결합니다. 별도 임베딩 코드 없이 OpenAI가 파싱/청킹/임베딩/검색을 모두 처리합니다.

- **셀 1 (초기 설정)**: Vector Store를 생성하고 `faqs.json`, `knowledgeBase.json`을 업로드 (최초 1회)
- **셀 2 (에이전트)**: `FileSearchTool(vector_store_ids=[vs_id])`를 도구로 추가해 에이전트 실행

In [6]:
# ── Vector Store 초기 설정 (최초 1회 실행) ──
# faqs.json / knowledgeBase.json 을 OpenAI Vector Store에 업로드합니다.
# 이미 같은 이름의 Vector Store가 있으면 재사용하고 재업로드하지 않습니다.

import openai
from pathlib import Path

client = openai.OpenAI()

VS_NAME = "notebook-faq-kb"
DATA_DIR = Path.cwd() / "data"

# 같은 이름의 Vector Store가 이미 있으면 재사용
existing = [vs for vs in client.vector_stores.list() if vs.name == VS_NAME]

if existing:
    vs_id = existing[0].id
    print(f"기존 Vector Store 재사용: {vs_id}")
else:
    vs = client.vector_stores.create(name=VS_NAME)
    vs_id = vs.id
    print(f"Vector Store 생성: {vs_id}")

    files_to_upload = [
        open(DATA_DIR / "faqs.json", "rb"),
        open(DATA_DIR / "knowledgeBase.json", "rb"),
    ]
    try:
        batch = client.vector_stores.file_batches.upload_and_poll(
            vector_store_id=vs_id,
            files=files_to_upload,
        )
        print(f"파일 업로드 완료 - 상태: {batch.status}  파일 수: {batch.file_counts.completed}")
    finally:
        for f in files_to_upload:
            f.close()

print(f"\nvs_id = '{vs_id}'")

Vector Store 생성: vs_699fe9cd84388191baf6c01f474cbbf1
파일 업로드 완료 - 상태: completed  파일 수: 2

vs_id = 'vs_699fe9cd84388191baf6c01f474cbbf1'


```
faqs.json
[
  {"question": "비밀번호를 어떻게 재설정하나요?", "answer": "비밀번호를 재설정하려면 계정 설정 페이지에서 '비밀번호 찾기'를 클릭하세요."},
  {"question": "환불 정책이 어떻게 되나요?", "answer": "구매일로부터 30일 이내 환불이 가능합니다. 주문 번호와 함께 고객 지원팀에 문의하세요."},
  {"question": "전화 상담을 제공하나요?", "answer": "네. 월요일부터 금요일 오전 10시부터 오후 6시(IST)까지 전화 상담이 가능합니다."},
  {"question": "이메일 주소는 어떻게 변경하나요?", "answer": "프로필 설정의 이메일 환경설정에서 이메일을 변경할 수 있습니다."},
  {"question": "제 데이터는 안전한가요?", "answer": "네. 모든 사용자 데이터는 업계 표준 보안 방식으로 암호화되어 보호됩니다."}
]
```

```
kowledgeBase.json
[
  {"question": "환불 정책이 어떻게 되나요?", "answer": "구매일로부터 30일 이내 모든 상품을 전액 환불받을 수 있습니다."},
  {"question": "주문은 어디서 조회하나요?", "answer": "웹사이트의 주문 조회 페이지에서 주문 번호로 조회할 수 있습니다."}
]

```

In [8]:
from agents import Agent, Runner, FileSearchTool, WebSearchTool
from openai.types.responses import ResponseTextDeltaEvent

# 대화 히스토리 저장 리스트 (멀티턴 대화를 위해 메시지를 누적)
messages = []

# 여행 및 FAQ 처리 에이전트 정의
agent = Agent(
    model="gpt-5-mini",
    name="여행·지식 베이스 에이전트",
    instructions=(
        "당신은 여행 계획과 회사 FAQ를 도와주는 에이전트입니다. "
        "날씨가 필요하면 get_weather, 최신 정보는 WebSearchTool을 사용하세요. "
        "계정·환불·정책·비밀번호·보안 등 FAQ 관련 질문은 file_search 도구로 "
        "지식 베이스를 검색한 뒤 답하세요. 답은 간결하게 하세요."
    ),
    tools=[
        get_weather,                               # 날씨 조회 도구 (이전 셀에서 정의)
        WebSearchTool(),                           # 최신 정보 웹 검색 도구
        FileSearchTool(vector_store_ids=[vs_id]),  # 회사 FAQ 지식 베이스 검색 도구
    ],
)

# 사용자가 'exit' 입력 시 종료되는 대화 루프
while True:
    user_input = input("\n사용자: ")
    if user_input == "exit":
        print("Bye")
        break

    # 사용자 메시지를 대화 히스토리에 추가
    messages.append({"role": "user", "content": user_input})

    print("\n에이전트: ", end="", flush=True)

    # 스트리밍 방식으로 에이전트 실행 (전체 대화 히스토리 전달)
    result = Runner.run_streamed(agent, input=messages)

    full_response = ""  # 스트리밍 응답 조각을 누적할 변수

    # 스트리밍 이벤트를 실시간으로 처리
    async for event in result.stream_events():
        # 텍스트 델타 이벤트만 필터링하여 출력
        if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
            delta = event.data.delta or ""
            print(delta, end="", flush=True)  # 글자 단위로 실시간 출력
            full_response += delta            # 전체 응답 누적

    # 완성된 에이전트 응답을 대화 히스토리에 추가
    messages.append({"role": "assistant", "content": full_response})


사용자:  환불 정책이 어떻게 되나요?



에이전트: 환불 정책: 구매일로부터 30일 이내 모든 상품은 전액 환불됩니다.   
절차: 주문 번호와 함께 고객 지원팀에 문의하시면 환불 처리됩니다. 

환불 신청을 도와드릴까요? 주문 번호를 알려 주세요.


사용자:  전화 상담을 제공하나요?



에이전트: 네 — 회사 FAQ에 따르면 전화 상담은 월요일~금요일 오전 10시부터 오후 6시(IST)에 제공됩니다. 

원하시면 이 시간을 귀하의 현지 시각으로 변환해 드리거나, 통화 예약을 도와드릴게요. 어느 시간대(또는 도시)로 변환해 드릴까요?


사용자:  exit


Bye


---------------
### 웹 UI 구현으로 연결

동일한 에이전트 구조를 **Streamlit 웹 앱**으로 구현한 예제가 **`01_OpenAI_API_Basic/10_chat-app-rag/app.py`** 입니다.
해당 앱에서는 채팅 화면, OpenAI로 답변 생성, 대화 기록 유지까지 포함합니다.

> 참고: `app.py`는 커스텀 임베딩(`rag.py`) 방식을 사용합니다. 위 노트북의 Vector Store 방식과 비교해 볼 수 있습니다.

실행: `10_chat-app-rag` 폴더에서 `streamlit run app.py`